In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses
import keras_nlp

In [ ]:
# Увеличим параметры для GPU
MAX_VOCAB = 10000     # Еще больше слов в словаре
CONTEXT_WIN = 50      # Длина контекста увеличена
EMBED_DIM = 512       # Более сложные эмбеддинги
HEADS = 8             # Степень двойки (8 голов по 64 размерности)
FEED_FORWARD = 2048   # В 4 раза больше EMBED_DIM (стандарт GPT)
TRANSFORMER_BLOCKS = 4 # Больше слоев
BATCH_SIZE = 64       # Больше батч для GPU
EPOCH = 40            # Эпох нужно меньше, так как данных много

# 1. Скачиваем датасет Шекспира
path = tf.keras.utils.get_file(
    'shakespeare.txt', 
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

with open(path, 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

# 2. Фильтруем короткие строки и добавляем токен конца 'endseq'
train_text = [line.strip() + ' endseq' for line in lines if len(line.strip()) > 15]

# Возьмем больше строк, так как у нас GPU
train_text = train_text[:20000] 

print(f"Количество строк для обучения: {len(train_text)}")
print(f"Пример данных: {train_text[0]}")

# 3. Настраиваем Subword токенизатор (Byte-Pair Encoding)
# Обучаем словарь на нашем тексте
# ВАЖНО: compute_word_piece_vocabulary ожидает tf.data.Dataset, а не список строк
vocab_data = tf.data.Dataset.from_tensor_slices(train_text)

vocab = keras_nlp.tokenizers.compute_word_piece_vocabulary(
    vocab_data,
    vocabulary_size=MAX_VOCAB,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[START]", "[END]"]
)

# Создаем сам токенизатор
tokenizer = keras_nlp.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=CONTEXT_WIN + 1,
)

def prep_data(train_text):
  # Токенизатор возвращает тензоры, нам нужно их обрезать
  seq = tokenizer(train_text)
  X_target = seq[:,:-1]
  y_label = seq[:,1:]
  return X_target, y_label

X_train, y_train = prep_data(train_text)

ValueError: Unknown value for `output_mode` argument of TextVectorization. Allowed values are: ('int', 'one_hot', 'multi_hot', 'count', 'tf_idf'). Received: output_mode=<class 'int'>

In [ ]:
def perplexity(true, pred):
  return tf.exp(tf.reduce_mean(losses.sparse_categorical_crossentropy(true, pred, from_logits=True)))

class TokenPositionEmbedding(layers.Layer):
  def __init__(self, context_win, max_vocab, embed_dim, **kwargs) -> None:
    super().__init__(**kwargs)
    self.token_embed = layers.Embedding(input_dim=max_vocab, output_dim=embed_dim)
    self.position_embed = layers.Embedding(input_dim=context_win, output_dim=embed_dim)

  def call(self, x):
    context_win = tf.shape(x)[-1]
    positions = self.position_embed(tf.range(start=0, limit=context_win, delta=1))

    return self.token_embed(x) + positions


NameError: name 'layers' is not defined

In [ ]:
class TransformerBlock(layers.Layer):
  def __init__(self, embed_dim, heads, feed_forward, rate=0.1, **kwargs) -> None:
    super().__init__(**kwargs)
    self.attention = layers.MultiHeadAttention(num_heads=heads, key_dim=embed_dim)
    self.feed_forward_net = models.Sequential([
        layers.Dense(feed_forward, activation='relu'),
        layers.Dense(embed_dim)
    ])

    self.norm1 = layers.LayerNormalization(epsilon=1e-6)
    self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    self.drop1 = layers.Dropout(rate)
    self.drop2 = layers.Dropout(rate)

  def call(self, inputs, training=False):
    attention_output = self.attention(inputs, inputs, use_causal_mask=True)
    attention_output = self.drop1(attention_output, training=training)
    output = self.norm1(inputs + attention_output)

    feed_forward_out = self.feed_forward_net(output)
    feed_forward_out = self.drop2(feed_forward_out, training=training)

    return self.norm2(feed_forward_out + output)


NameError: name 'layers' is not defined

In [ ]:
class LLM(models.Model):
  def __init__(self, context_win, max_vocab, embed_dim, heads, feed_forward, num_transformer_blocks, batch_size, epoch, **kwargs):
    super().__init__(**kwargs)
    self.embed_layer = TokenPositionEmbedding(context_win, max_vocab, embed_dim)
    self.blocks = [
        TransformerBlock(embed_dim, heads, feed_forward) for _ in range(num_transformer_blocks)
    ]
    self.dense_out = layers.Dense(max_vocab)

  def call(self, inputs, training=False):
    x = self.embed_layer(inputs)
    for block in self.blocks:
      x = block(x, training=training)
    return self.dense_out(x)

def gen(model, prompt, length=30, temperature=0.8, top_k=10, repetition_penalty=1.2):
  input_tensor = tokenizer([prompt])
  tokens = [token for token in input_tensor.numpy()[0] if token != 0]

  gen_text = prompt

  for _ in range(length):
    # Берем только последние CONTEXT_WIN токенов
    context_tok = tokens[-CONTEXT_WIN:]
    input_data = tf.convert_to_tensor([context_tok])

    preds = model(input_data, training=False)
    next_logits = preds[0, -1, :].numpy() # Переводим в numpy для изменения
    
    # --- ШТРАФ ЗА ПОВТОРЕНИЯ (Repetition Penalty) ---
    for tok in set(tokens):
        if next_logits[tok] < 0:
            next_logits[tok] *= repetition_penalty
        else:
            next_logits[tok] /= repetition_penalty
    # ------------------------------------------------
    
    # Применяем температуру
    next_logits = next_logits / (temperature + 1e-7)
    
    # Возвращаем в тензор для top_k
    next_logits_tf = tf.convert_to_tensor(next_logits)
    
    # Выбираем top_k вероятных слов
    top_vals, top_ind = tf.math.top_k(next_logits_tf, k=top_k)
    top_probs = tf.nn.softmax(top_vals).numpy()

    # Случайный выбор из top_k с учетом их вероятностей
    next_ind = np.random.choice(top_ind.numpy(), p=top_probs)
    
    # Защита от генерации паддинга (0)
    if next_ind == 0 and len(top_ind.numpy()) > 1:
      next_ind = top_ind.numpy()[1]

    # Получаем слово из словаря
    next_word = tokenizer.vocabulary[next_ind]
    
    # Остановка при токене конца
    if next_word == 'endseq':
        break

    tokens.append(next_ind)
    
    # Обработка subword токенов (убираем ##)
    if next_word.startswith('##'):
        gen_text += next_word[2:]
    else:
        gen_text += ' ' + next_word

  return gen_text

NameError: name 'models' is not defined

In [ ]:
def main():
  test_model = LLM(
      CONTEXT_WIN,
      len(vocab),
      EMBED_DIM,
      HEADS,
      FEED_FORWARD,
      TRANSFORMER_BLOCKS,
      BATCH_SIZE,
      EPOCH
  )

  test_model.compile(
      optimizer='adam',
      loss = losses.SparseCategoricalCrossentropy(from_logits = True),
      metrics = [perplexity]
  )

  test_model.fit(
      x=X_train, y=y_train,
      batch_size = BATCH_SIZE,
      epochs = EPOCH,
      verbose=1
  )

  test_input1 = input('\nEnter first text to test: ')
  test_input2 = input('Enter second text to test: ')

  print(f'\nPrompt : {test_input1}\nGenerated text: {gen(test_model, test_input1, length=30)}')
  print(f'\nPrompt : {test_input2}\nGenerated text: {gen(test_model, test_input2, length=30)}')

main()